
## Patch Instruction for `AutoPipeline` Before Running MASE Pipeline

To prevent the `TypeError: object of type 'NoneType' has no len()` error, follow these steps:

1. **Open the file**: ./mase/src/chop/pipelines/auto_pipeline.py

2. **Locate line 26** inside the `AutoPipeline.__init__` method.

3. **Replace the existing line** with:

```python
self.pass_outputs = [{}] * len(self.pass_groups)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as T
import torchvision.datasets as datasets
import torchvision.models as models

from chop import MaseGraph
import chop.passes as passes
from chop.pipelines.auto_pipeline import AutoPipeline
from pathlib import Path

# ------------------------------
# 1️⃣ Device setup
# ------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------
# 2️⃣ CIFAR-10 dataloaders
# ------------------------------
transform = T.Compose([
    T.Resize(224),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],
                [0.229, 0.224, 0.225])
])
train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
val_dataset   = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64, shuffle=False)

# ------------------------------
# 3️⃣ Load pre-trained ResNet18
# ------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 10)  # CIFAR-10 has 10 classes
model = model.to(device)

# ------------------------------
# 4️⃣ Quick fine-tuning (FP32)
# ------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
epochs = 1  # short demo; increase for better accuracy

for epoch in range(epochs):
    model.train()
    running_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            _, preds = model(imgs).max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total

acc_fp32 = evaluate(model, val_loader)
print(f"✅ FP32 Accuracy: {acc_fp32:.2f}%")

# ------------------------------
# 5️⃣ Create MASE Graph
# ------------------------------
example_inputs = torch.randn(1, 3, 224, 224).to(device)
dummy_inputs = {"x": example_inputs}

mg = MaseGraph(model)
mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(mg, pass_args={"dummy_in": dummy_inputs})
print("✅ Vision MASE Graph created.")

# ------------------------------
# 6️⃣ Setup output directory and pipeline
# ------------------------------
output_dir = Path("./mase_output")
output_dir.mkdir(exist_ok=True)
pipeline = AutoPipeline(run_training=True)

# ------------------------------
# 7️⃣ PTQ-only
# ------------------------------
pass_args_ptq = {
    "dummy_in": dummy_inputs,
    "quantize": True,
    "qat": False,  # PTQ only
    "output_dir": str(output_dir)
}

ptq_graph, _ = pipeline(mg, pass_args=pass_args_ptq)
ptq_model = torch.fx.GraphModule(ptq_graph.model, ptq_graph.fx_graph).to(device)
acc_ptq = evaluate(ptq_model, val_loader)
print(f"✅ PTQ-only Accuracy: {acc_ptq:.2f}%")

# ------------------------------
# 8️⃣ PTQ + QAT
# ------------------------------
pass_args_qat = {
    "dummy_in": dummy_inputs,
    "quantize": True,
    "qat": True,   # enable QAT fine-tuning
    "output_dir": str(output_dir)
}

qat_graph, _ = pipeline(mg, pass_args=pass_args_qat)
qat_model = torch.fx.GraphModule(qat_graph.model, qat_graph.fx_graph).to(device)
acc_qat = evaluate(qat_model, val_loader)
print(f"✅ PTQ + QAT Accuracy: {acc_qat:.2f}%")

/home/clingcl0ng/new_mase_workspace/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/clingcl0ng/new_mase_workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Epoch 1/1, Loss: 0.3268
✅ FP32 Accuracy: 93.26%
✅ Vision MASE Graph created.
✅ PTQ-only Accuracy: 93.26%
✅ PTQ + QAT Accuracy: 93.26%


In [2]:
# ------------------------------
# 8️⃣ Optional: Export to ONNX
# ------------------------------
torch.onnx.export(
    qat_model,
    example_inputs,
    str(output_dir / "resnet18_int8.onnx"),
    opset_version=17,
    input_names=["x"],
    output_names=["output"],
    dynamic_axes={"x": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"✅ INT8 ONNX model exported to {output_dir / 'resnet18_int8.onnx'}")

✅ INT8 ONNX model exported to mase_output/resnet18_int8.onnx
